In [1]:
import os
import json
import pandas as pd
import numpy as np
import torch
import faiss
import ollama

from sentence_transformers import SentenceTransformer

print("CUDA:", torch.cuda.is_available())

CUDA: True


In [2]:
df = pd.read_csv(
    "../data/processed/m_absa_selected.csv"
)

reviews = (
    df[["text", "language", "domain"]]
    .drop_duplicates(subset=["text"])
    .reset_index(drop=True)
)

train_reviews = (
    reviews[
        reviews["text"].isin(
            df[df["split"] == "train"]["text"].unique()
        )
    ]
    .reset_index(drop=True)
)

index = faiss.read_index(
    "../vector_store/index.faiss"
)

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Reviews:", len(reviews))
print("Train reviews:", len(train_reviews))
print("FAISS vectors:", index.ntotal)
print("Embedding device:", embedding_model.device)

Reviews: 17795
Train reviews: 10355
FAISS vectors: 10355
Embedding device: cuda:0


In [3]:
def retrieve_reviews(query, top_k=5, domain=None):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    search_k = max(top_k * 5, 20)

    scores, indices = index.search(
        query_embedding,
        search_k
    )

    results = train_reviews.iloc[indices[0]].copy()
    results["score"] = scores[0]

    if domain is not None:
        results = results[
            results["domain"] == domain
        ]

    return (
        results
        .sort_values("score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

In [4]:
def build_context(results):

    parts = []

    for i, row in results.iterrows():

        parts.append(
            f"[Review {i + 1}]\n"
            f"Language: {row['language']}\n"
            f"Domain: {row['domain']}\n"
            f"Similarity: {row['score']:.3f}\n"
            f"Text: {row['text']}"
        )

    return "\n\n".join(parts)

In [5]:
def generate_llm_only(query):

    prompt = f"""
You are a review intelligence assistant.

Analyze the following customer-review query.

Query:
{query}

Return ONLY valid JSON:

{{
    "main_finding": "...",
    "common_opinion": "...",
    "negative_signals": [],
    "key_aspects": [],
    "conclusion": "..."
}}

Rules:
- Answer based only on your general language understanding.
- You do NOT have access to the review database.
- Do NOT invent specific customer evidence.
- Keep the answer concise.
"""

    response = ollama.chat(
        model="qwen3:8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.message.content

In [6]:
def generate_rag(query, domain, top_k=5):

    results = retrieve_reviews(
        query=query,
        top_k=top_k,
        domain=domain
    )

    context = build_context(results)

    prompt = f"""
You are a review intelligence assistant.

Analyze the query using ONLY the retrieved customer reviews.

Query:
{query}

Retrieved reviews:
{context}

Return ONLY valid JSON:

{{
    "main_finding": "...",
    "common_opinion": "...",
    "negative_signals": [],
    "key_aspects": [],
    "evidence": [],
    "conclusion": "..."
}}

Rules:
- Use only the retrieved reviews as evidence.
- Do not invent customer opinions.
- Do not make claims unsupported by the retrieved reviews.
- If there are no negative signals, return [].
- Include 2-3 short evidence statements.
"""

    response = ollama.chat(
        model="qwen3:8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.message.content, results

In [7]:
evaluation_queries = [
    {
        "id": "hotel_en",
        "query": "The hotel room was very clean",
        "language": "en",
        "domain": "hotel"
    },
    {
        "id": "hotel_ru",
        "query": "Номер в отеле был очень чистым",
        "language": "ru",
        "domain": "hotel"
    },
    {
        "id": "hotel_de",
        "query": "Das Hotelzimmer war sehr sauber",
        "language": "de",
        "domain": "hotel"
    },
    {
        "id": "laptop_en",
        "query": "The laptop battery drains very quickly",
        "language": "en",
        "domain": "laptop"
    },
    {
        "id": "laptop_ru",
        "query": "Батарея ноутбука быстро разряжается",
        "language": "ru",
        "domain": "laptop"
    },
    {
        "id": "laptop_de",
        "query": "Der Akku des Laptops hält nicht lange",
        "language": "de",
        "domain": "laptop"
    },
    {
        "id": "restaurant_en",
        "query": "The food was delicious but the service was slow",
        "language": "en",
        "domain": "restaurant"
    },
    {
        "id": "restaurant_ru",
        "query": "Еда была вкусной, но обслуживание было медленным",
        "language": "ru",
        "domain": "restaurant"
    },
    {
        "id": "restaurant_de",
        "query": "Das Essen war lecker, aber der Service war langsam",
        "language": "de",
        "domain": "restaurant"
    }
]

In [8]:
evaluation_results = []

for item in evaluation_queries:

    print("=" * 100)
    print(item["id"])
    print("Language:", item["language"])
    print("Domain:", item["domain"])
    print("Query:", item["query"])

    # ==========================================================
    # 1. LLM ONLY
    # ==========================================================

    print("\n[1] LLM ONLY")

    llm_answer = generate_llm_only(
        item["query"]
    )

    print(llm_answer)

    # ==========================================================
    # 2. RAG
    # ==========================================================

    print("\n[2] RAG")

    rag_answer, retrieved = generate_rag(
        query=item["query"],
        domain=item["domain"],
        top_k=5
    )

    print(rag_answer)

    # ==========================================================
    # 3. Save results
    # ==========================================================

    evaluation_results.append({
        "id": item["id"],
        "query": item["query"],
        "language": item["language"],
        "domain": item["domain"],
        "llm_only": llm_answer,
        "rag": rag_answer,
        "top_similarity": retrieved["score"].iloc[0],
        "mean_similarity": retrieved["score"].mean()
    })

    print("\n")

{
    "main_finding": "Der Service wurde als langsam bewertet, während das Essen positiv aufgenommen wurde.",
    "common_opinion": "Der Service war langsam, was die Kundenerfahrung beeinträchtigte.",
    "negative_signals": [
        "Der Service war langsam, ich musste warten, bis ich bestellen und Essen bekommen konnte, obwohl es nicht voll war.",
        "Service war slow had to wait to order and get food although not crowded.",
        "aber der Service war etwas langsam."
    ],
    "key_aspects": ["Servicegeschwindigkeit"],
    "evidence": [
        "Der Service war langsam, ich musste warten, bis ich bestellen und Essen bekommen konnte, obwohl es nicht voll war.",
        "Service was slow had to wait to order and get food although not crowded.",
        "aber der Service war etwas langsam."
    ],
    "conclusion": "Die Bewertungen zeigen eindeutig, dass der Service als langsamer wahrgenommen wurde, was die Gesamtbewertung negativ beeinflusste."
}




In [9]:
evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df.to_csv(
    "../data/processed/rag_llm_evaluation.csv",
    index=False,
    encoding="utf-8"
)

evaluation_df

,id,query,language,domain,llm_only,rag,top_similarity,mean_similarity
0,hotel_en,The hotel room was very clean,en,hotel,"{\n ""main_finding"": ""The hotel room was ver...","{\n ""main_finding"": ""The hotel rooms are co...",0.924704,0.904311
1,hotel_ru,Номер в отеле был очень чистым,ru,hotel,"{\n ""main_finding"": ""Cleanliness of the roo...","{\n ""main_finding"": ""Rooms were clean"",\n ...",0.931051,0.912820
2,hotel_de,Das Hotelzimmer war sehr sauber,de,hotel,"{\n ""main_finding"": ""Das Hotelzimmer war se...","{\n ""main_finding"": ""Die Hotelzimmer werden...",0.952393,0.922003
3,laptop_en,The laptop battery drains very quickly,en,laptop,"{\n ""main_finding"": ""The laptop battery dra...","{\n ""main_finding"": ""No direct evidence of ...",0.774869,0.749937
4,laptop_ru,Батарея ноутбука быстро разряжается,ru,laptop,"{\n ""main_finding"": ""Problems with battery ...","{\n ""main_finding"": ""Отсутствие прямых отзы...",0.818506,0.791360
5,laptop_de,Der Akku des Laptops hält nicht lange,de,laptop,"{\n ""main_finding"": ""The laptop's battery d...","{\n ""main_finding"": ""Der Akkubetrag des Lap...",0.877214,0.851295
6,restaurant_en,The food was delicious but the service was slow,en,restaurant,"{\n ""main_finding"": ""Mixed experience with ...","{\n ""main_finding"": ""Mixed feedback with po...",0.806453,0.735556
7,restaurant_ru,"Еда была вкусной, но обслуживание было медленным",ru,restaurant,"{\n ""main_finding"": ""Mixed review with both...","{\n ""main_finding"": ""The user's query highl...",0.768614,0.740947
8,restaurant_de,"Das Essen war lecker, aber der Service war lan...",de,restaurant,"{\n ""main_finding"": ""Mixed experience with ...","{\n ""main_finding"": ""Der Service wurde als ...",0.831964,0.782598


In [10]:
retrieval_stats = evaluation_df[
    [
        "language",
        "domain",
        "top_similarity",
        "mean_similarity"
    ]
].copy()

print(
    retrieval_stats.groupby("language")[
        ["top_similarity", "mean_similarity"]
    ].mean()
)

print()

print(
    retrieval_stats.groupby("domain")[
        ["top_similarity", "mean_similarity"]
    ].mean()
)

          top_similarity  mean_similarity
language                                 
de              0.887190         0.851965
en              0.835342         0.796601
ru              0.839390         0.815042

            top_similarity  mean_similarity
domain                                     
hotel             0.936049         0.913045
laptop            0.823530         0.797530
restaurant        0.802344         0.753034


In [11]:
from IPython.display import display


def interactive_demo(query, domain=None, top_k=5):

    print("=" * 100)
    print("REVIEWLENS AI — INTERACTIVE DEMO")
    print("=" * 100)

    print("\nUSER QUERY:")
    print(query)

    # ==========================================================
    # 1. LLM ONLY
    # ==========================================================

    print("\n" + "=" * 100)
    print("1. LLM ONLY — WITHOUT RETRIEVED CONTEXT")
    print("=" * 100)

    llm_answer = generate_llm_only(query)

    print(llm_answer)

    # ==========================================================
    # 2. RAG RETRIEVAL
    # ==========================================================

    print("\n" + "=" * 100)
    print("2. FAISS RETRIEVAL")
    print("=" * 100)

    rag_answer, retrieved = generate_rag(
        query=query,
        domain=domain,
        top_k=top_k
    )

    display(
        retrieved[
            ["text", "language", "domain", "score"]
        ].style.format(
            {"score": "{:.3f}"}
        )
    )

    # ==========================================================
    # 3. CONTEXT
    # ==========================================================

    context = build_context(retrieved)

    print("\n" + "=" * 100)
    print("3. CONTEXT SENT TO QWEN")
    print("=" * 100)

    print(context)

    # ==========================================================
    # 4. RAG ANSWER
    # ==========================================================

    print("\n" + "=" * 100)
    print("4. RAG + QWEN")
    print("=" * 100)

    print(rag_answer)

    # ==========================================================
    # 5. PIPELINE
    # ==========================================================

    print("\n" + "=" * 100)
    print("PIPELINE")
    print("=" * 100)

    print("""
LLM ONLY:
Query → Qwen → Answer

RAG:
Query → Multilingual → Embedding → FAISS → Top-K Reviews → Context → Qwen → Evidence-based → Answer
""")

    return {
        "query": query,
        "llm_only": llm_answer,
        "retrieved_reviews": retrieved,
        "context": context,
        "rag_answer": rag_answer
    }

In [12]:
demo = interactive_demo(
    query="The restaurant food was good but the service was slow",
    domain="restaurant",
    top_k=5
)


3. CONTEXT SENT TO QWEN
[Review 1]
Language: en
Domain: restaurant
Similarity: 0.792
Text: Service was slow had to wait to order and get food although not crowded .

[Review 2]
Language: de
Domain: restaurant
Similarity: 0.726
Text: Der Service war langsam, ich musste warten, bis ich bestellen und Essen bekommen konnte, obwohl es nicht voll war.

[Review 3]
Language: ru
Domain: restaurant
Similarity: 0.714
Text: Обслуживание было медленным, пришлось подождать, чтобы сделать заказ и получить еду, хотя народу было немного.

[Review 4]
Language: de
Domain: restaurant
Similarity: 0.679
Text: Das Restaurant ist nett, aber nicht gehoben.

[Review 5]
Language: ru
Domain: restaurant
Similarity: 0.674
Text: меню выглядело великолепно, а официант был очень любезен, но когда принесли еду, она оказалась посредственной.

4. RAG + QWEN
{
    "main_finding": "The restaurant had slow service, but food quality was mixed.",
    "common_opinion": "Service was slow despite low customer volume.",
    "neg

In [13]:
demo = interactive_demo(
    query="Батарея ноутбука быстро разряжается",
    domain="laptop",
    top_k=5
)


3. CONTEXT SENT TO QWEN
[Review 1]
Language: ru
Domain: laptop
Similarity: 0.819
Text: зарядное устройство для ноутбука неоднократно искрило.

[Review 2]
Language: ru
Domain: laptop
Similarity: 0.800
Text: 5 фунтов ноутбук с девятью часами работы время работы аккумулятора.

[Review 3]
Language: en
Domain: laptop
Similarity: 0.788
Text: 5 pound laptop with its nine hour battery life .

[Review 4]
Language: de
Domain: laptop
Similarity: 0.776
Text: 5 Pfund Laptop mit neun Stunden Akkulaufzeit.

[Review 5]
Language: de
Domain: laptop
Similarity: 0.774
Text: Akkulaufzeit: Ich habe den Laptop noch nicht so lange, aber mit einer Ladung läuft er problemlos 8 Stunden oder mehr.

4. RAG + QWEN
{
    "main_finding": "Отзывы не подтверждают проблему быстрого разряжения батареи, в отличие от заявленного в запросе.",
    "common_opinion": "Большинство отзывов указывают на длительное время работы аккумулятора (до 9 часов).",
    "negative_signals": [],
    "key_aspects": ["время работы аккумулятора

In [15]:
demo = interactive_demo(
    query="  завтраки очень вкусные, но работа персонала оставляет жеалпть лучшего",
    domain="hotel",
    top_k=5
)


3. CONTEXT SENT TO QWEN
[Review 1]
Language: de
Domain: hotel
Similarity: 0.825
Text: Sehr lecker das Frühstücksbuffet.

[Review 2]
Language: de
Domain: hotel
Similarity: 0.825
Text: Frühstücksbuffet nicht so gut, und Lage vielleicht etwas daneben.

[Review 3]
Language: ru
Domain: hotel
Similarity: 0.813
Text: Завтрак отличный.

[Review 4]
Language: de
Domain: hotel
Similarity: 0.812
Text: Auch Frühstück sehr schön!

[Review 5]
Language: ru
Domain: hotel
Similarity: 0.809
Text: Очень хорошо завтрак.

4. RAG + QWEN
{
    "main_finding": "Позитивные отзывы о качестве завтраков, отсутствие упоминаний о работе персонала",
    "common_opinion": "Завтраки оцениваются как отличные или очень вкусные",
    "negative_signals": [],
    "key_aspects": ["качество завтраков", "работа персонала (отсутствует в отзывах)"],
    "evidence": [
        "Завтрак отличный (Review 3)",
        "Очень хорошо завтрак (Review 5)",
        "Sehr lecker das Frühstücksbuffet (Review 1)"
    ],
    "conclusion": "О